# Day 06 — Data Preprocessing & Feature Engineering

**Notebook-style learning notes + hands-on examples**

Completed topics:
1. Why preprocessing is needed
2. Train / Validation / Test Split
3. Data Leakage
4. Advanced Missing-Value Handling
5. Categorical Encoding
6. One-Hot Encoding
7. Label / Ordinal Encoding
8. Feature Scaling
9. Standardization
10. Min-Max Normalization
11. Robust Scaling
12. Outliers

> Golden rule: **FIT on training data → TRANSFORM training/test data using the same learned transformation.**


## 1. Why Data Preprocessing?

Real-world data can contain:

- Missing values
- Categorical values
- Different numerical scales
- Outliers
- Data leakage

Typical ML workflow:

**Raw data → Clean → Encode → Scale → Handle outliers → Train model**


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "Age": [22, 25, 30, 35, 40],
    "Salary": [30000, 45000, 50000, 65000, 80000],
    "City": ["Hyderabad", "Chennai", "Vizag", "Hyderabad", "Chennai"],
    "Experience": ["Beginner", "Intermediate", "Advanced", "Intermediate", "Advanced"]
})

df


## 2. Train / Validation / Test Split

- **Training data:** model learns parameters.
- **Validation data:** tune hyperparameters / select models.
- **Test data:** final evaluation on unseen data.

For preprocessing, split first and then fit transformations only on training data.


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Salary"])
y = df["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


## 3. Data Leakage

Leakage happens when information from outside the training process enters the model.

### Wrong
```python
scaler.fit_transform(X)   # before splitting
```

The scaler learns statistics from the entire dataset, including test data.

### Correct
```python
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
```

The same idea applies to imputers, feature selection, PCA, etc.


## 4. Missing Values

Common strategies:

| Situation | Typical approach |
|---|---|
| Numerical, symmetric | Mean |
| Numerical, skewed/outliers | Median |
| Categorical | Mode |
| Time series | ffill / bfill |
| Similar rows available | KNN |
| Missingness is informative | Missing indicator |

Never calculate imputation statistics using the full dataset before splitting.


In [ ]:
missing_df = pd.DataFrame({
    "Salary": [30000, 35000, 40000, 45000, 500000, np.nan],
    "City": ["Hyderabad", "Chennai", "Vizag", np.nan, "Chennai", "Hyderabad"]
})

print("Missing values:")
print(missing_df.isnull().sum())

# Median for skewed numerical data
missing_df["Salary"] = missing_df["Salary"].fillna(
    missing_df["Salary"].median()
)

# Mode for categorical data
missing_df["City"] = missing_df["City"].fillna(
    missing_df["City"].mode()[0]
)

missing_df


## 5. Categorical Encoding

Categorical features must be represented numerically for many traditional ML algorithms.

### Nominal
No meaningful order:
- City
- Color
- Blood group

### Ordinal
Meaningful order:
- Small < Medium < Large
- Beginner < Intermediate < Advanced


## 6. One-Hot Encoding

Use mainly for **nominal categories**.

Example:

`Red, Blue, Green`

becomes:

```text
Red   → 1 0 0
Blue  → 0 1 0
Green → 0 0 1
```

No artificial ordering is introduced.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

city_encoded = encoder.fit_transform(df[["City"]])

encoded_city_df = pd.DataFrame(
    city_encoded,
    columns=encoder.get_feature_names_out(["City"])
)

encoded_city_df


## 7. Label Encoding vs Ordinal Encoding

For a binary category, 0/1 representation is natural:

`No → 0`, `Yes → 1`

For ordered categories, use **Ordinal Encoding**:

`Beginner → 1`, `Intermediate → 2`, `Advanced → 3`

> Note: In scikit-learn, `LabelEncoder` is primarily intended for target labels (`y`). For input features (`X`), use `OneHotEncoder` or `OrdinalEncoder` as appropriate.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

experience_order = [["Beginner", "Intermediate", "Advanced"]]

ordinal_encoder = OrdinalEncoder(
    categories=experience_order
)

experience_encoded = ordinal_encoder.fit_transform(
    df[["Experience"]]
)

experience_encoded


## 8. Feature Scaling

Scaling brings numerical features to comparable scales.

Especially important for:
- KNN
- K-Means
- SVM
- PCA
- Many gradient-based algorithms

Usually not required for:
- Decision Trees
- Random Forests
- Other tree-based models


## 9. Standardization

Standardization gives a feature approximately:

- Mean = 0
- Standard deviation = 1

Formula:

**z = (x − μ) / σ**

It uses the **mean and standard deviation**, so it can be affected by outliers.


In [ ]:
from sklearn.preprocessing import StandardScaler

X_num = df[["Age", "Salary"]]

standard_scaler = StandardScaler()

X_standardized = standard_scaler.fit_transform(X_num)

standardized_df = pd.DataFrame(
    X_standardized,
    columns=["Age_standardized", "Salary_standardized"]
)

standardized_df


## 10. Min-Max Normalization

Min-Max scaling usually maps values to **0–1**.

Formula:

**x' = (x − xmin) / (xmax − xmin)**

Example:

For `[10, 20, 30, 40, 50]`:

- 10 → 0
- 50 → 1

It is sensitive to outliers because it uses the minimum and maximum.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

X_normalized = minmax_scaler.fit_transform(X_num)

normalized_df = pd.DataFrame(
    X_normalized,
    columns=["Age_normalized", "Salary_normalized"]
)

normalized_df


## 11. Robust Scaling

Robust Scaling is useful when outliers are present.

It uses:

- **Median**
- **IQR = Q3 − Q1**

Formula:

**x' = (x − median) / IQR**

It is more robust to extreme values than Standardization and Min-Max scaling.

Important: RobustScaler does **not remove outliers**. It only makes scaling less sensitive to them.


In [ ]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()

X_robust = robust_scaler.fit_transform(X_num)

robust_df = pd.DataFrame(
    X_robust,
    columns=["Age_robust", "Salary_robust"]
)

robust_df


## 12. Outliers

An outlier is a value unusually far from most observations.

### IQR method

**IQR = Q3 − Q1**

**Lower bound = Q1 − 1.5 × IQR**

**Upper bound = Q3 + 1.5 × IQR**

Values outside these bounds are potential outliers.

> Detection does not mean automatic removal. First decide whether the value is valid, unusual but meaningful, or a data error.


In [ ]:
salary = pd.Series([30000, 35000, 40000, 45000, 50000, 500000])

Q1 = salary.quantile(0.25)
Q3 = salary.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = salary[
    (salary < lower_bound) | (salary > upper_bound)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Potential outliers:")
print(outliers)


# Quick Revision

| Technique | Main idea |
|---|---|
| One-Hot Encoding | Nominal categories → binary columns |
| Ordinal Encoding | Ordered categories → ordered numbers |
| Standardization | Mean 0, SD 1 |
| Min-Max | Usually 0–1 |
| Robust Scaling | Median + IQR |
| IQR Outlier Detection | Q1/Q3 and 1.5 × IQR |
| Data Leakage Prevention | Fit transformations on training data only |

## Next Topics

13. Feature Selection  
14. Feature Engineering  
15. Pipelines  
16. ColumnTransformer  
17. Imbalanced Data  
18. SMOTE  
19. End-to-End Preprocessing Pipeline
